# Wrist Fracture / Abnormality Detection from X-rays
### Medical Imaging ML Project — Notebook Documentation

## Task 0: Scope Declaration

- **Body part / anatomical region:** Wrist (upper extremity)
- **Imaging modality:** X-ray (plain radiograph)
- **Problem type:** Classification (binary)
- **Target classes:** `normal` vs. `abnormal` (abnormal = any fracture, hardware, degenerative change, or other musculoskeletal abnormality visible on the study, per the original radiologist labels)
- **Dataset source:** [MURA: Musculoskeletal Radiographs, Stanford ML Group](https://stanfordmlgroup.github.io/competitions/mura/) — filtered to the `XR_WRIST` subset. Accessed: **`<< fill in your download date >>`**
- **Real-world question:** Could an automated screening tool flag wrist X-rays that are likely abnormal, so they can be prioritised for radiologist review?
- **Who would use this:** In principle, a triage assistant for radiologists or emergency-department clinicians reviewing a queue of wrist X-rays — **not** a replacement for radiologist diagnosis (see A1 and A8).

> **Note on scope:** MURA covers seven upper-extremity body parts (elbow, finger, forearm, hand, humerus, shoulder, wrist). This project deliberately scopes to **wrist only** to keep the dataset size and training time manageable for a coursework timeline. The code below is written so that changing a single variable (`STUDY_TYPE`) would let you re-run the same pipeline on any other MURA body part.


In [1]:
# --- Environment setup: run this first ---
# Installs PyTorch/torchvision (CPU build by default) plus the other libraries
# this notebook needs. Safe to re-run; pip skips packages that are already installed.
# If you have an NVIDIA GPU and want CUDA acceleration, replace the torch/torchvision
# line below with the command for your CUDA version from https://pytorch.org/get-started/locally/
import sys

%pip install -q torch torchvision
%pip install -q pandas scikit-learn matplotlib pillow

print("Setup complete. Python:", sys.version)


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Setup complete. Python: 3.10.12 (main, Feb 15 2026, 10:59:58) [GCC 13.3.0]


In [2]:
# Config — change this one variable to point at your own MURA download
import os

MURA_ROOT = "./MURA-v1.1"          # << set this to wherever you extracted MURA-v1.1
STUDY_TYPE = "XR_WRIST"            # << change to re-run on a different body part
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4
RANDOM_SEED = 42

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


ModuleNotFoundError: No module named 'torch'

## Task 0a: Dataset Sourcing

- **Dataset:** MURA v1.1 (Musculoskeletal Radiographs), Stanford ML Group.
- **Access:** Obtained via Stanford AIMI's dataset catalogue on Redivis (stanford.redivis.com), after requesting and being granted data access to "MURA: MSK Xrays". (MURA can alternatively be obtained directly from the [original MURA page](https://stanfordmlgroup.github.io/competitions/mura/), which requires filling in a short form to receive a download link — both distribute the identical dataset and folder structure.) Licence permits academic/research/coursework use (see the MURA paper and dataset page for the exact terms — cite these in your final submission).
- **Access date:** `<< fill in the date you were granted access / downloaded the files >>`
- **Size:** ~40,561 images from 14,863 musculoskeletal studies across 7 body parts. The wrist subset alone is ~9,752 images from ~3,697 studies.
- **Structure:** Images are organised as `train/<BODY_PART>/patientNNNNN/studyN_<positive|negative>/imageN.png`, plus `train_image_paths.csv`, `train_labeled_studies.csv`, `valid_image_paths.csv`, `valid_labeled_studies.csv`. The `positive`/`negative` suffix in the study folder name **is the label** (positive = abnormal).
- **Why this dataset fits the assignment:** it is public, requires only a free registration (consistent with the brief's allowed-dataset list), is large enough to split into train/val/test without leakage at the **patient/study** level, and every image already has a verified radiologist-assigned label — no re-labelling needed.


## A1. Problem Definition & Clinical Framing

**Task:** Given a single wrist X-ray image, classify it as **normal** or **abnormal**.

- **Positive / abnormal** means the original MURA radiologist labelling identified *some* abnormality in that study — this is a broad "not normal" bucket covering fractures, hardware (e.g. plates, screws), degenerative changes, and other visible abnormalities. It does **not** distinguish between abnormality types.
- **Negative / normal** means the radiologist found no abnormality in that study.
- A positive prediction should be read as *"this image resembles studies that a radiologist flagged as abnormal"* — not as a specific diagnosis (e.g. it does not say "this is a scaphoid fracture").

**This is a coursework prototype, not a diagnostic tool.**
- It has not been validated on any clinical population, has not undergone regulatory review, and is trained on a single (US academic hospital system) data source.
- It must not be used to make or support any actual clinical decision, and is not safe to run on real patient images outside this course.
- It is intended purely to demonstrate the ML pipeline: data handling, model training, evaluation, and a working inference demo.

**Assumptions about input images:**
- Single 2D static wrist radiograph (PNG), one view per image (MURA includes multiple views per study — this model scores each image independently, not per-study).
- Reasonable diagnostic image quality — the model has not been trained on, and is not expected to handle well, heavily rotated, cropped, non-wrist, or non-radiograph images (see A8 and the Streamlit app's uncertainty handling).


## A2. Dataset Documentation

Load the official MURA CSVs for the wrist subset and summarise class balance and basic image properties.


In [ ]:
import pandas as pd
import glob

def find_studies(split_root, study_type):
    """Scan the filesystem directly — no CSV needed. MURA's folder naming
    already encodes the label: a study folder ending in "_positive" is
    abnormal, "_negative" is normal. This matches the exact folder structure
    MURA is distributed in, including the Stanford AIMI / Redivis export."""
    pattern = os.path.join(split_root, study_type, "patient*", "study*_*")
    rows = []
    for study_dir in sorted(glob.glob(pattern)):
        folder_name = os.path.basename(study_dir)
        if folder_name.endswith("positive"):
            label = 1
        elif folder_name.endswith("negative"):
            label = 0
        else:
            continue
        patient_id = os.path.basename(os.path.dirname(study_dir))
        rows.append({"study_path": study_dir, "label": label, "patient_id": patient_id})
    return pd.DataFrame(rows)

train_studies = find_studies(os.path.join(MURA_ROOT, "train"), STUDY_TYPE)
valid_studies = find_studies(os.path.join(MURA_ROOT, "valid"), STUDY_TYPE)

assert len(train_studies) > 0 and len(valid_studies) > 0, (
    f"No studies found for {STUDY_TYPE} under {MURA_ROOT}/train or /valid — "
    "check MURA_ROOT points at the folder containing train/ and valid/ directly."
)

print(f"{STUDY_TYPE} — official train studies: {len(train_studies)}, official valid studies: {len(valid_studies)}")
print("\nTrain label distribution (0=normal, 1=abnormal):")
print(train_studies.label.value_counts())
print("\nValid label distribution:")
print(valid_studies.label.value_counts())


In [ ]:
import matplotlib.pyplot as plt

counts = train_studies.label.value_counts().sort_index()
plt.figure(figsize=(4,3))
plt.bar(["Normal (0)", "Abnormal (1)"], counts.values, color=["#4C72B0", "#C44E52"])
plt.title(f"{STUDY_TYPE} — class distribution (train studies)")
plt.ylabel("Number of studies")
plt.show()


**Write-up (fill in with your actual numbers once you've run the cells above):**

- Source: MURA v1.1, wrist subset, official train/valid split as released by Stanford ML Group.
- Size: `<< N train studies, N valid studies, N total images >>` — fill in from the printed output.
- Class distribution: `<< e.g. train is roughly 55/45 normal/abnormal, mild imbalance >>` — note this in A6 when you report confusion matrix / per-class metrics.
- Image format: PNG, grayscale radiographs; original dimensions vary by acquisition — resized to 224×224 for the model (see A3).
- Labelling: labels come from the original MURA release, assigned by board-certified radiologists per the dataset's methodology paper; not re-verified by us.
- Known limitations: MURA is sourced from a single US hospital system (Stanford Hospital), so demographic and equipment skew relative to other populations is a real limitation (expand on this in A8). Some body parts/classes in the full MURA set are more imbalanced than others — check the wrist-specific numbers above rather than assuming the overall MURA balance applies.
- Licence: MURA is released for research/academic use following registration on the Stanford ML Group page; confirm you have read and are complying with the exact terms stated there before submitting.


## A3. Data Preparation & Splitting

**Splitting strategy:** MURA is already organised so that all images from a given study/patient sit in one folder. We split at the **study level** (not the image level) to guarantee no leakage: multiple images from the same study never end up in different splits.

- **Train / Validation:** derived from the official MURA `train` set, split 85% / 15% by study, stratified by label.
- **Test:** the official MURA `valid` set is used as our held-out **test** set. It was never touched during training or model selection, so it gives an honest, leakage-free performance estimate.

We explicitly check that no `patientNNNNN` ID appears in more than one of our three splits.


In [ ]:
from sklearn.model_selection import train_test_split

# patient_id was already extracted in find_studies() above
unique_patients = train_studies["patient_id"].unique()
train_patients, val_patients = train_test_split(
    unique_patients, test_size=0.15, random_state=RANDOM_SEED
)

train_df = train_studies[train_studies.patient_id.isin(train_patients)].reset_index(drop=True)
val_df   = train_studies[train_studies.patient_id.isin(val_patients)].reset_index(drop=True)
test_df  = valid_studies.copy()  # official MURA "valid" set = our test set

# Leakage check: no patient ID should appear in more than one split
train_p, val_p, test_p = set(train_df.patient_id), set(val_df.patient_id), set(test_df.patient_id)
assert not (train_p & val_p), "Leakage between train and val!"
assert not (train_p & test_p), "Leakage between train and test!"
assert not (val_p & test_p), "Leakage between val and test!"
print("No patient overlap between train / val / test splits — confirmed.")
print(f"Train studies: {len(train_df)} | Val studies: {len(val_df)} | Test studies: {len(test_df)}")


NameError: name 'train_studies' is not defined

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import glob

def studies_to_image_list(df):
    """Expand each study row into one row per image inside that study folder."""
    rows = []
    for _, r in df.iterrows():
        for img_path in sorted(glob.glob(os.path.join(r.study_path, "*.png"))):
            rows.append({"image_path": img_path, "label": int(r.label)})
    return pd.DataFrame(rows)

train_images_df = studies_to_image_list(train_df)
val_images_df   = studies_to_image_list(val_df)
test_images_df  = studies_to_image_list(test_df)

print(f"Images — train: {len(train_images_df)}, val: {len(val_images_df)}, test: {len(test_images_df)}")

# --- Preprocessing pipeline: THIS MUST MATCH streamlit_app.py EXACTLY ---
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),   # MURA images are grayscale; backbone expects 3 channels
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),         # augmentation: wrist X-rays are reasonably left-right symmetric
    transforms.RandomRotation(degrees=7),           # augmentation: small tolerance for acquisition angle variation
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

class MuraDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row.image_path).convert("L")
        img = self.transform(img)
        return img, row.label

train_ds = MuraDataset(train_images_df, train_transform)
val_ds   = MuraDataset(val_images_df, eval_transform)
test_ds  = MuraDataset(test_images_df, eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


**Write-up:**

- Cleaning/resizing: images converted to grayscale then replicated to 3 channels (required by the ImageNet-pretrained backbone), resized to 224×224.
- Normalisation: ImageNet mean/std, since we use an ImageNet-pretrained ResNet backbone.
- Augmentation (train only): random horizontal flip and ±7° rotation, chosen to be mild — wrist X-rays have limited valid orientation variation, so aggressive augmentation (e.g. large rotations, colour jitter) would create unrealistic images.
- Split: study/patient-level 85/15 train/val from official MURA train; official MURA valid used as test. **Confirmed programmatically above that no patient ID appears in more than one split.**


## A4. Model Architecture & Justification

**Architecture:** ResNet18 pretrained on ImageNet, with the final fully-connected layer replaced for binary classification.

**Why this fits:** this is a binary image classification problem, not detection or segmentation, so a standard CNN classifier backbone is the natural fit — no bounding-box heads or pixel-decoder needed. A pretrained backbone is used because MURA-wrist (~8k training images) is small relative to what's needed to train a deep CNN from scratch; transfer learning from ImageNet features (edges, textures, shapes) gives a much stronger starting point.

- **Input shape:** `(batch, 3, 224, 224)` — 3-channel replicated grayscale, 224×224 (ResNet's standard input size).
- **Output shape:** `(batch, 1)` — a single logit, passed through a sigmoid to give the probability of "abnormal".
- **Alternatives considered:**
  - *Training a CNN from scratch* — rejected: too few images for the model to learn good low-level features from scratch within a coursework timeline.
  - *A larger backbone (ResNet50)* — considered, but ResNet18 was chosen first as a faster-to-train baseline; ResNet50 is a reasonable next experiment if ResNet18's ceiling is reached (note in A6 whether you tried this).
  - *Freezing all backbone layers vs. fine-tuning* — we fine-tune the whole network (not just the final layer) since X-ray textures differ enough from natural ImageNet images that adapting earlier layers helps; if training is unstable, freezing early layers is the fallback.


In [ ]:
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 1)   # single logit for binary classification
model = model.to(device)

print(model.fc)


## A5. Training Process

- **Loss:** `BCEWithLogitsLoss` — appropriate for binary classification with a single output logit.
- **Optimiser:** Adam.
- **Learning rate:** see `LEARNING_RATE` in the config cell (start at 1e-4; lower if loss is unstable).
- **Batch size / epochs:** see config cell.
- **Overfitting controls:** augmentation (A3), a validation set checked every epoch, and early stopping on validation loss (patience below). Dropout was not added on top of ResNet18 initially — if train/val curves diverge sharply, add `nn.Dropout` before the final `fc` layer as a next step.


In [ ]:
import torch.optim as optim
import copy

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_loss = float("inf")
best_model_state = None
patience, patience_counter = 3, 0

def run_epoch(loader, training):
    model.train() if training else model.eval()
    total_loss, correct, n = 0.0, 0, 0
    torch.set_grad_enabled(training)
    for imgs, labels in loader:
        imgs = imgs.to(device)
        labels = labels.float().unsqueeze(1).to(device)
        if training:
            optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        if training:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        preds = (torch.sigmoid(logits) > 0.5).float()
        correct += (preds == labels).sum().item()
        n += imgs.size(0)
    return total_loss / n, correct / n

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = run_epoch(train_loader, training=True)
    val_loss, val_acc = run_epoch(val_loader, training=False)
    history["train_loss"].append(train_loss); history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc); history["val_acc"].append(val_acc)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | train_loss={train_loss:.4f} acc={train_acc:.3f} "
          f"| val_loss={val_loss:.4f} acc={val_acc:.3f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1} (no val_loss improvement for {patience} epochs).")
            break

model.load_state_dict(best_model_state)  # restore best checkpoint before evaluation


ModuleNotFoundError: No module named 'torch'

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()

axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.tight_layout(); plt.show()


NameError: name 'plt' is not defined

**Write-up (fill in after running):** describe what the curves actually show — e.g. does val loss track train loss closely (good) or diverge upward while train loss keeps falling (overfitting — and if so, say whether early stopping caught it, or whether you'd add dropout/more augmentation next)?

## A6. Evaluation & Results

Evaluated on the **held-out test set** (official MURA `valid` split for wrist) — never seen during training or model selection.


In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)
import numpy as np

model.eval()
all_labels, all_probs = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        logits = model(imgs)
        probs = torch.sigmoid(logits).cpu().numpy().flatten()
        all_probs.extend(probs)
        all_labels.extend(labels.numpy())

all_labels = np.array(all_labels)
all_probs = np.array(all_probs)
all_preds = (all_probs > 0.5).astype(int)

print(f"Accuracy:  {accuracy_score(all_labels, all_preds):.3f}")
print(f"Precision: {precision_score(all_labels, all_preds):.3f}")
print(f"Recall:    {recall_score(all_labels, all_preds):.3f}")
print(f"F1:        {f1_score(all_labels, all_preds):.3f}")
print()
print(classification_report(all_labels, all_preds, target_names=["normal", "abnormal"]))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(4,4))
plt.imshow(cm, cmap="Blues")
plt.xticks([0,1], ["normal","abnormal"]); plt.yticks([0,1], ["normal","abnormal"])
plt.xlabel("Predicted"); plt.ylabel("True")
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i,j], ha="center", va="center")
plt.title("Confusion Matrix (test set)")
plt.colorbar()
plt.show()


NameError: name 'model' is not defined

In [ ]:
# At least one correct and one incorrect example, shown as images
correct_idx = np.where(all_preds == all_labels)[0]
incorrect_idx = np.where(all_preds != all_labels)[0]

panels = []
if len(correct_idx) > 0:
    panels.append((correct_idx[0], "Correct"))
else:
    print("No correctly classified test examples to show.")
if len(incorrect_idx) > 0:
    panels.append((incorrect_idx[0], "Incorrect"))
else:
    print("No misclassified test examples to show (perfect accuracy on this test set).")

if panels:
    fig, axes = plt.subplots(1, len(panels), figsize=(4 * len(panels), 4))
    if len(panels) == 1:
        axes = [axes]
    for ax, (idx, title) in zip(axes, panels):
        img_path = test_images_df.iloc[idx].image_path
        img = Image.open(img_path).convert("L")
        ax.imshow(img, cmap="gray")
        ax.set_title(f"{title}\nTrue: {'abnormal' if all_labels[idx] else 'normal'} | "
                     f"Pred: {'abnormal' if all_preds[idx] else 'normal'} "
                     f"({all_probs[idx]:.2f})")
        ax.axis("off")
    plt.tight_layout(); plt.show()


NameError: name 'all_preds' is not defined

**Write-up (fill in with your actual numbers):**

- Report the printed accuracy/precision/recall/F1 and discuss them **together**, not accuracy alone — e.g. if precision and recall diverge, say what that means for false positives vs false negatives in this context.
- Look at the confusion matrix: is one class being systematically confused for the other? Does that match the class imbalance you found in A2?
- Look at your incorrect example: does it share any visible trait (positioning, hardware, image quality) that might explain the mistake? Be specific and honest — "the model struggles with X because Y" is more valuable than "the model isn't perfect."


## A7. Model Export

Save the trained weights and record the exact preprocessing pipeline that the Streamlit app must reproduce **exactly**.


In [ ]:
import torch, torchvision

torch.save(model.state_dict(), "wrist_abnormality_resnet18.pt")

print("Saved to wrist_abnormality_resnet18.pt")
print("torch version used to save:", torch.__version__)
print("torchvision version used:", torchvision.__version__)


ModuleNotFoundError: No module named 'torch'

**Preprocessing pipeline the app must reapply exactly (see `eval_transform` above, and `streamlit_app.py`):**

1. Convert the uploaded image to grayscale.
2. Replicate to 3 channels.
3. Resize to 224×224.
4. Convert to tensor (0–1 range).
5. Normalise with ImageNet mean `[0.485, 0.456, 0.406]` and std `[0.229, 0.224, 0.225]`.
6. No augmentation at inference time (no flips/rotations — those are train-only).

Architecture to reload: `torchvision.models.resnet18` with `model.fc` replaced by `nn.Linear(in_features, 1)`, weights loaded from `wrist_abnormality_resnet18.pt`, output passed through `sigmoid` to get a probability.


## A8. Limitations, Ethics & Next Steps

**Generalisation limits:**
- MURA was collected at a single US academic hospital system, on its specific X-ray equipment. Performance is not validated on other scanners, hospitals, countries, or age groups (MURA skews adult; paediatric wrist X-rays look meaningfully different and this model has not seen them).
- The model only ever saw the "positive = some abnormality" label, not abnormality subtypes, so it cannot and must not be read as identifying a specific condition.

**Ethical considerations:**
- **False negatives** (missing an abnormal case) are the more clinically dangerous failure mode here — a missed fracture flagged as "normal" could delay treatment. **False positives** cause unnecessary follow-up but are lower-stakes.
- Given this asymmetry, a real deployment would need the decision threshold and the reviewing workflow to be tuned to minimise false negatives specifically — not just optimise for overall accuracy — and would need a human radiologist reviewing every flagged case regardless of model confidence.
- This tool must never be used as a substitute for radiologist review; at most it could support triage/prioritisation, always with clinical oversight.

**What would be needed for real clinical use (not achieved by this coursework project):**
- External validation on data from other hospitals/populations/scanners.
- Regulatory clearance (e.g. FDA/CE marking as appropriate) as a medical device.
- Prospective clinical evaluation with radiologists, not just retrospective accuracy on a held-out split.
- A much larger and more diagnostically specific label set if the goal is anything beyond coarse triage.


---
## Notebook Setup Instructions

1. Register and download **MURA v1.1** from https://stanfordmlgroup.github.io/competitions/mura/
2. Extract it so you have a `MURA-v1.1/` folder containing `train/`, `valid/`, and the four CSV files.
3. Set `MURA_ROOT` in the config cell at the top to that folder's path.
4. Run all cells top to bottom. Training (A5) will take a while on CPU — a GPU (e.g. free Colab GPU runtime) is strongly recommended.
5. After running A7, copy `wrist_abnormality_resnet18.pt` into the same folder as `streamlit_app.py`.
